# Example 1: Computing p-values with Adaptive CoRT-SI

This notebook demonstrates how to use the `cort_si` package to compute selective inference p-values for Adaptive CoRT-SI.

In [7]:
import random
import sys

import numpy as np

sys.path.append('..')

from cort_si import SI, SI_randj, gen_data

## 1. Generate Synthetic Data

In [2]:
np.random.seed(0)
random.seed(0)

p = 5
s = 2
nS = 6
nT = 7
true_beta = 1.0
num_info_aux = 1
num_uninfo_aux = 1
gamma = 0.05

XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, beta0 = gen_data.generate_data(
    p=p,
    s=s,
    nS=nS,
    nT=nT,
    true_beta=true_beta,
    num_info_aux=num_info_aux,
    num_uninfo_aux=num_uninfo_aux,
    gamma=gamma,
)

print(f'Number of auxiliary groups: {len(XS_list)}')
print(f'Feature dimension: {p}')
print(f'Target sample size: {X0.shape[0]}')

Number of auxiliary groups: 2
Feature dimension: 5
Target sample size: 7


## 2. Set Parameters

In [3]:
lambda_sel = 0.05
lambda0 = 0.05
lambdak_list = [0.05] * len(XS_list)
T = 3
z_min = -5
z_max = 5

print(f'lambda_sel: {lambda_sel}')
print(f'lambda0: {lambda0}')
print(f'T: {T}')

lambda_sel: 0.05
lambda0: 0.05
T: 3


## 3. Compute p-values for All Selected Features

In [8]:
p_values = SI(
    X0,
    Y0,
    XS_list,
    YS_list,
    lambda_sel=lambda_sel,
    lambda0=lambda0,
    lambdak_list=lambdak_list,
    SigmaS_list=SigmaS_list,
    Sigma0=Sigma0,
    T=T,
    z_min=z_min,
    z_max=z_max,
)

if p_values is not None:
    print(f'Number of selected features: {len(p_values)}')
    print('\nFeature index and p-values:')
    for j, p_val in p_values:
        print(f'Feature {j}: beta0[{j}] = {beta0[j]:.4f}, p-value = {p_val:.4f}')
else:
    print('No features selected')

Number of selected features: 5

Feature index and p-values:
Feature 0: beta0[0] = 1.0000, p-value = 0.0614
Feature 1: beta0[1] = 1.0000, p-value = 0.0533
Feature 2: beta0[2] = 0.0000, p-value = 0.7578
Feature 3: beta0[3] = 0.0000, p-value = 0.6979
Feature 4: beta0[4] = 0.0000, p-value = 0.8358


## 4. Compute a p-value for a Random Selected Feature

This section reuses the same synthetic-data setup and parameter choice, then applies `SI_randj(...)` to one randomly chosen selected target feature.

In [9]:
rng_seed = 11

if p_values is None:
    print('No features selected')
else:
    selected_features = [j for j, _ in p_values]
    j_rand = random.Random(rng_seed).choice(selected_features)
    random.seed(rng_seed)
    p_value_rand = SI_randj(
        X0,
        Y0,
        XS_list,
        YS_list,
        lambda_sel=lambda_sel,
        lambda0=lambda0,
        lambdak_list=lambdak_list,
        SigmaS_list=SigmaS_list,
        Sigma0=Sigma0,
        T=T,
        z_min=z_min,
        z_max=z_max,
    )

    print(f'Number of selected features: {len(selected_features)}')
    print(f'Selected feature set: {selected_features}')
    if p_value_rand is not None:
        print(f'Random selected feature: {j_rand}')
        print(f'true beta0[{j_rand}] = {beta0[j_rand]:.4f}')
        print(f'p-value = {p_value_rand:.4f}')
    else:
        print('No valid truncation interval for the random selected feature')

Number of selected features: 5
Selected feature set: [0, 1, 2, 3, 4]
Random selected feature: 3
true beta0[3] = 0.0000
p-value = 0.6979
